In [ ]:
%load_ext autoreload
%autoreload 2
%aimport -tqdm, -matplotlib, -torch, -torchinfo, -numpy, -timm, -cv2, -rich, -torchvision
# !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

from tqdm import tqdm
import matplotlib
import os
import sys

if "PAPERMILL_EXECUTION" in os.environ:
    # matplotlib.use('Agg')
    is_papermill = True
else:
    is_papermill = False

import torch
import torch.nn as nn
import timm
import matplotlib.pyplot as plt
import numpy as np
import cv2
from IPython.display import HTML, Video, Image
from rich.pretty import pprint
from numpy.typing import NDArray
import torchvision

import pyvista as pv
# pv.global_theme.trame.jupyter_extension_enabled = True

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))

import utils
import torchinfo
from pathlib import Path
from utils.torch.datasets import QueriedFaceDataset
import model as my_model
from utils.torch.viz import display_sample
%matplotlib inline

In [ ]:
import data as my_data
import json
from utils.datasets import CanonicalLandmarks, VideoDataset
from utils.torch.datasets import sample_random_clips
from data import queries_68_ibug, queries_70_synth, queries_98_wflw, queries_opt, face_mesh, DatasetName

In [ ]:
QueriedFaceDataset.unregister_all()

datasets_dir = Path("/home/dogsch/dev/datasets")
datasets = my_data.Datasets(datasets_dir)

print(datasets)

In [ ]:
from utils.datasets.base import ImageDataset
import utils.torch
from itertools import islice
from utils.torch.viz import display_sample



for i in range(1):
    data = datasets.wflw
    # data = dataloader
    i = np.random.randint(0, len(data))
    sample_idx = int(i)
    sample = data[sample_idx]
    sample = QueriedFaceDataset.wrap(sample)[:1, :2]
    display_sample(sample, show_queries=True, show_weights=False, sample_idx=i)
    # sample = QueriedFaceDataset.wrap(sample)
    # display_sample(sample[0], show_queries=True, show_weights=True, sample_idx=i)
    # display_sample(sample[1], show_queries=True, show_weights=True, sample_idx=i)

In [ ]:
# import train
# dataloaders = train.DataLoaders(datasets, None)

In [ ]:
# %matplotlib inline
# display_sample(next(dataloaders.wflw), show_weights=False, show_queries=True)
# s = next(dataloaders.wflw)
# s["labels"].shape, QueriedFaceDataset.wrap(s).labels.shape

In [ ]:
if False:
    dest_dir.mkdir(parents=True, exist_ok=True)
    for i, batch in enumerate(tqdm(test_clips, total=256)):
        clip_idx = i
        
        # is_flipped = batch.get_is_flipped()[0].item() # type: ignore
        # assert not is_flipped, "Expected no flips in saved clips"
        
        for j in range(8):
            img = (batch.images[j].cpu().numpy().transpose(1, 2, 0) * 255).round().clip(0, 255).astype(np.uint8)
            lbl = (batch.labels[j].cpu().numpy())
            
            dest_img_path = dest_dir / f"clip_{clip_idx:03d}_frame_{j:02d}.png"
            dest_lbl_path = dest_dir / f"clip_{clip_idx:03d}_frame_{j:02d}_lbl.npy"
            
            cv2.imwrite(str(dest_img_path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            np.save(dest_lbl_path, lbl)

In [ ]:
fe = timm.create_model(
    "hgnetv2_b1.ssld_stage1_in22k_in1k",
    pretrained= False,
    out_indices=[-4, -3, -2, -1],
    features_only=True
)

print(torchinfo.summary(fe, input_size=[(1, 3, 224, 224)], col_names=("input_size", "output_size", "num_params", "trainable"), row_settings=("var_names", "depth"), depth=1))

In [ ]:
fe = my_model.ImageFeatureExtractor(global_dim=96, d_model=96, n_heads=[2,3,4])
print(torchinfo.summary(fe, input_size=[(1, 3, 224, 224)], col_names=("input_size", "output_size", "num_params", "trainable"), row_settings=("var_names", "depth"), depth=1))

## Model Instantiation

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from utils.torch.misc import Config, save, load, optimizer_to
import utils.torch.optim
from torch.optim.lr_scheduler import CyclicLR

Cov2D = my_model.LowRankCov2D
LandmarkPrediction = my_model.LandmarkPrediction

synth_clip_len = 3
config = Config(
    name="infinite-lmk-v4",
    run="bypass-attention19_5",
    learning_rate=1.5e-4,
    batch_size=24 * synth_clip_len,
    best_nme=None,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
global_step = 0

try:
    del model  # type: ignore
except:
    pass

model = my_model.QLOT(feature_extractor_pretrained=False)
optimizer = None
scheduler = None

# Function to adapt old checkpoint keys to new ones
def weights_filter(state_dict: dict) -> dict:
    if "gating_tau" in state_dict:
        state_dict["gating_radius"] = state_dict.get("gating_tau", torch.tensor(1.0))
        del state_dict["gating_tau"]
    elif "tau_param" in state_dict:
        state_dict["gating_radius"] = state_dict.get("tau_param", torch.tensor(1.0))
        del state_dict["tau_param"]
        
    state_dict = my_model.QLOT.translate_weights(state_dict)

    return state_dict

model_queries = my_data.make_query_points()

def load_queries(state_dict: dict):
    model_queries.load_state_dict(state_dict)

extra_save_args = {
    "model_func": weights_filter,
    "query_points": model_queries,
}

global_step, config = load("../../models/qlot-final.pth", model, None, scheduler, **extra_save_args)

model.to(device)
# optimizer_to(optimizer, device)

config, global_step

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider, FloatSlider
from model.update_predictor import PhaseModulatedPE

def visualize_pmpe(pe: PhaseModulatedPE) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    pe: PhaseModulatedPE = pe.to("cpu").eval()
    plt.figure()

    num_freqs = pe.num_freqs
    tau = pe.tau
    beta = pe.beta

    if pe.in_dims != 1:
        assert pe.in_dims == 3
        fig, axes = plt.subplots(2, 2, figsize=(14, 6))
        axes[0, 0].plot(pe.omega.detach()[0].cpu().numpy())
        axes[1, 0].plot(pe.omega.detach()[1].cpu().numpy())
        axes[1, 1].plot(pe.omega.detach()[2].cpu().numpy())
        axes[0, 1].plot(pe.phi.detach().cpu().numpy())
        fig.show()
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].plot(pe.omega.detach()[0].cpu().numpy())
        axes[1].plot(pe.phi.detach().cpu().numpy())
        fig.show()
    
    n_samples = 400
    x = torch.linspace(-1.0, 1.0, n_samples).unsqueeze(-1)  # (n_samples, 1)
    with torch.no_grad():
        feats = pe(x).numpy()  # (n_samples, out_dims)
    # Drop the raw-x passthrough channel so the heatmap shows only the spectral features.
    # feats = feats[:, 1:]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: feature channels vs 1D input.
    im0 = axes[0].imshow(
        feats.T,
        aspect="auto",
        origin="lower",
        extent=[-1, 1, 0, feats.shape[1]],
        cmap="RdBu",
        vmin=-feats.max(),
        vmax=feats.max(),
    )
    axes[0].set_xlabel("input x")
    axes[0].set_ylabel("feature channel")
    axes[0].set_title(f"PMPE features (num_freqs={num_freqs}, tau={tau:.4g}, beta={beta:.3f})")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # Right: dot-product kernel between encoded 1D values.
    with torch.no_grad():
        gram = feats @ feats.T  # (n_samples, n_samples)
    im1 = axes[1].imshow(
        gram,
        aspect="auto",
        origin="lower",
        extent=[-1, 1, -1, 1],
        cmap="viridis",
    )
    axes[1].set_xlabel("input x")
    axes[1].set_ylabel("input x")
    axes[1].set_title("dot-product kernel ⟨PE(x), PE(x')⟩")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()

    return pe.omega.detach().cpu().numpy().copy(), pe.phi.detach().cpu().numpy().copy(), feats.copy()

if True:
    @interact(
        num_fourier_freqs=IntSlider(value=16, min=1, max=64, step=1, description="num_fourier_freqs"),
        num_phase_mod_freqs=IntSlider(value=8, min=1, max=64, step=1, description="num_phase_mod_freqs"),
        tau=FloatLogSlider(value=0.01, base=10, min=-3, max=0, step=0.01, description="tau"),
    )
    def visualize_pmpe_interactive(num_fourier_freqs: int, num_phase_mod_freqs: int, tau: float, beta: float):
        pe = PhaseModulatedPE(in_dims=1, num_fourier_freq=num_fourier_freqs, num_phase_mod_freq=num_phase_mod_freqs, tau=tau).eval()
        visualize_pmpe(pe)

In [ ]:
import torchinfo
print(torchinfo.summary(model, input_size=[(1, 3, 224, 224), (1, 98, 3)], col_names=("input_size", "output_size", "num_params", "trainable"), row_settings=("var_names", "depth"), return_tensor_predictions=True, iterations=2))

## Visualize Query Points

In [ ]:
import ipywidgets as widgets
import pyvista as pv
import numpy as np
import torch
from dataclasses import dataclass

try:
    del plotter  # type: ignore
except:
    pass

pv.set_jupyter_backend("server")
plotter = pv.Plotter(notebook=True, window_size=[1280, 1024])
query_points = my_data.queries_opt.cpu().clone()
query_points.requires_grad_(False)
log_output = widgets.Output()

@dataclass
class PickerState:
    selected_idx: tuple[str, int] | None
    points: dict[str, NDArray[np.float32]]
    actors: dict[str, pv.Actor]
    picked_point: pv.Actor | None

state = PickerState(
    selected_idx=None,
    points={},
    actors={},
    picked_point=None
)

keys = sorted(query_points.queries.keys())
for key in keys:
    state.points[key] = query_points.queries[key].detach().cpu().numpy()
    state.actors[key] = plotter.add_points(
        state.points[key],
        render_points_as_spheres=True,
        point_size=10,
        color=plt.colormaps.get("tab10").colors[len(state.actors) % 10], # type: ignore
        pickable=True,
        label=key
    )

# Add mesh
plotter.add_mesh(face_mesh, color="lightgrey", opacity=0.5, pickable=False)
plotter.add_mesh(face_mesh, style="wireframe", opacity=0.1, pickable=False)

# Widgets
lbl_info = widgets.Label("Click a point to select")

# Sliders
sx = widgets.FloatSlider(description='X', min=-1, max=1, step=0.00001, disabled=True, readout_format='.5f')
sx.layout.width = "1200px"
sy = widgets.FloatSlider(description='Y', min=-1.5, max=1, step=0.00001, disabled=True, readout_format='.5f')
sy.layout.width = "1200px"
sz = widgets.FloatSlider(description='Z', min=-1.6, max=0.7, step=0.00001, disabled=True, readout_format='.5f')
sz.layout.width = "1200px"

btn_snap = widgets.Button(description="Snap to Surface", disabled=True)

def update_point(change=None):
    idx = state.selected_idx
    if idx is None:
        return
    key, idx = idx
    
    new_pos = np.array([sx.value, sy.value, sz.value])
    
    # Update or add picked point actor
    assert state.picked_point is not None
    state.picked_point.mapper.SetInputData(pv.PolyData(new_pos.reshape(1, 3)))
    plotter.update()
    
    
sx.observe(update_point, names='value')
sy.observe(update_point, names='value')
sz.observe(update_point, names='value')

def snap(b):
    idx = state.selected_idx
    if idx is None:
        return
    key, idx = idx
    
    pt = np.array([sx.value, sy.value, sz.value])
    _, closest_pt = face_mesh.find_closest_cell(pt, return_closest_point=True) # type: ignore
    
    qp = query_points.get(key)
    qp[idx] = torch.as_tensor(closest_pt)
    
    flip_idx = query_points.canonical_landmarks[key].flip_horizontal_indices()
    qp[flip_idx[idx]] = torch.as_tensor(closest_pt * [-1, 1, 1])  # Flip X for the corresponding point

    query_points.symmetrize()
    for k, actor in state.actors.items():
        state.points[k] = query_points.queries[k].detach().cpu().numpy()
        actor.mapper.SetInputData(pv.PolyData(state.points[k]))
    closest_pt = state.points[key][idx]
    
    # Update sliders (will trigger update_point)
    sx.value = closest_pt[0]
    sy.value = closest_pt[1]
    sz.value = closest_pt[2]
    
    plotter.update()
    
btn_snap.on_click(snap)

def on_pick(point):
    try:
        # Find closest point in cloud to the picked coordinate
        pids = [
            (key, actor.mapper.dataset.find_closest_point(point))
            for key, actor in state.actors.items() if actor.pickable
        ]
        pid = min(pids, key=lambda x: np.linalg.norm(state.points[x[0]][x[1]] - point))
        key, idx = pid
        state.selected_idx = (key, idx)
        
        lbl_info.value = f"Selected: {key} ({idx})"
        pt = state.points[key][idx]
        
        sx.unobserve(update_point, names='value')
        sy.unobserve(update_point, names='value')
        sz.unobserve(update_point, names='value')
        
        sx.disabled = False
        sy.disabled = False
        sz.disabled = False
        btn_snap.disabled = False
        
        sx.value = pt[0]
        sy.value = pt[1]
        sz.value = pt[2]
        
        sx.observe(update_point, names='value')
        sy.observe(update_point, names='value')
        sz.observe(update_point, names='value')
        
        if state.picked_point is not None:
            plotter.remove_actor(state.picked_point) # type: ignore

        state.picked_point = plotter.add_points(
            pt.reshape(1, 3),
            render_points_as_spheres=True,
            point_size=20,
            color='yellow',
            name="selection",
            pickable=False
        )
    except Exception as e:
        log_output.append_stdout(str(e))
    
plotter.enable_point_picking(callback=on_pick, show_message=False, color='pink', point_size=20)

display(widgets.VBox([lbl_info, sx, sy, sz, btn_snap, log_output]))
plotter.show()

In [ ]:
torch.save(query_points.state_dict(), "edited_query_points.pth")

In [ ]:
def select_key_idx(key: str, idx: int):
    state.selected_idx = (key, idx)
    lbl_info.value = f"Selected: {key} ({idx})"
    pt = state.points[key][idx]
    
    sx.unobserve(update_point, names='value')
    sy.unobserve(update_point, names='value')
    sz.unobserve(update_point, names='value')
    
    sx.disabled = False
    sy.disabled = False
    sz.disabled = False
    btn_snap.disabled = False
    
    sx.value = pt[0]
    sy.value = pt[1]
    sz.value = pt[2]
    
    sx.observe(update_point, names='value')
    sy.observe(update_point, names='value')
    sz.observe(update_point, names='value')

    if state.picked_point is not None:
        plotter.remove_actor(state.picked_point) # type: ignore

    state.picked_point = plotter.add_points(
        pt.reshape(1, 3),
        render_points_as_spheres=True,
        point_size=20,
        color='yellow',
        name="selection",
        pickable=False
    )
select_key_idx(DatasetName.WFLW, 60)

## Evaluate on 300-W test set

In [ ]:
dataloader = torch.utils.data.DataLoader(torch.utils.data.ConcatDataset([datasets.ibug_test_common]), batch_size=64, shuffle=False, drop_last=False, num_workers=4)

In [ ]:
iod_indices_ibug = (36, 45)  # Indices for left and right eye corners in 68-point markup

model.eval()

nme_vals = []
for i, batch in enumerate(dataloader):
    images = batch["image"].to(device)
    queries = model_queries.get(DatasetName.Ibug).to(device).expand(images.shape[0], -1, -1)
    
    with torch.no_grad():
        preds: LandmarkPrediction
        preds = model(images, queries, iterations=3)
    
        xy = preds.mean.cpu()
        labels = batch["labels"].cpu()
        
        if i == (1):
            # Plot 4 samples in grid
            fig, axes = plt.subplots(3, 3, figsize=(15, 15))
            for ax_j, j in enumerate(range(9)):
                ax = axes[ax_j // 3, ax_j % 3]
                img = batch["image"][j].cpu().permute(1, 2, 0).numpy()
                ax.imshow(img)
                ax.scatter(labels[j, :, 0], labels[j, :, 1], c='green', s=10, label='Ground Truth')
                ax.scatter(xy[j, :, 0], xy[j, :, 1], c='red', s=10, label='Predicted')
                ax.set_title(f'Sample {j}')
        
        errors = torch.sqrt(torch.sum((xy - labels) ** 2, dim=-1))  # (batch_size, num_queries)
        
        # Normalize by inter-ocular distance (distance between eye corners)
        xy_iod = xy[:, iod_indices_ibug, :]  # (batch_size, 2, 2)
        norm_factor = torch.sqrt(torch.sum((xy_iod[:, 0, :] - xy_iod[:, 1, :]) ** 2, dim=-1))  # (batch_size,)
        nme = (errors / norm_factor.unsqueeze(-1)).mean(dim=-1).tolist()
        nme_vals.extend(nme)
nme_vals = np.array(nme_vals)
nme_vals = nme_vals[nme_vals < 0.1]
print(f"Test NME: {np.mean(nme_vals):.6f} ± {np.std(nme_vals):.6f}, max={np.max(nme_vals):.6f}, min={np.min(nme_vals):.6f}")

print(nme_vals[np.argsort(-nme_vals)][:10])

## Evaluate on WFLW Test set

In [ ]:
checkpoint_nmes = {}
checkpoint_nmfs = {}

In [ ]:
iod_indices_wflw = (60, 72)  # Indices for left and right eye corners in 98-point markup

dataloader = torch.utils.data.DataLoader(datasets.wflw_test_full, batch_size=64, shuffle=False, drop_last=False, num_workers=4)
model.eval()

checkpoints = "/home/dogsch/dev/pa-face-landmark-tracking/.cache/models/infinite-lmk-v4/checkpoints/write-read-mixer15_2"
checkpoint_paths = utils.list_files_in_dir(checkpoints, pattern="*.pth")
checkpoint_paths.sort()

checkpoint_iter = tqdm(checkpoint_paths)

for checkpoint_path in checkpoint_iter:
    stem = Path(checkpoint_path).stem
    try:
        stem_int = int(stem)
    except:
        stem_int = 100000
    if stem_int < 50000 or stem_int in checkpoint_nmes:
        continue
    if "queries" in stem:
        continue

    global_step, config = load(checkpoint_path, model, None, None, **extra_save_args)

    nme_vals = []
    for i, batch in enumerate(dataloader):
        images = batch["image"].to(device)
        # queries = batch["queries"].to(device)
        
        with torch.no_grad():
            wflw_queries = model_queries.queries["WFLW"].to(device).unsqueeze(0).expand(images.shape[0], -1, -1)
            preds: LandmarkPrediction
            preds = model(images, wflw_queries, iterations=3)
        
            xy = preds.mean.cpu()
            labels = batch["labels"].cpu()
            
            if i == 0 and False:
                # Plot 4 samples in grid
                fig, axes = plt.subplots(2, 2, figsize=(10, 10))
                for j in range(4):
                    ax = axes[j // 2, j % 2]
                    img = batch["image"][j].cpu().permute(1, 2, 0).numpy()
                    ax.imshow(img)
                    ax.scatter(labels[j, :, 0], labels[j, :, 1], c='green', s=10, label='Ground Truth')
                    ax.scatter(xy[j, :, 0], xy[j, :, 1], c='red', s=10, label='Predicted')
                    ax.set_title(f'Sample {j}')
                fig.show()
            
            errors = torch.sqrt(torch.sum((xy - labels) ** 2, dim=-1))  # (batch_size, num_queries)
            
            # Normalize by inter-ocular distance (distance between eye corners)
            xy_iod = xy[:, iod_indices_wflw, :]  # (batch_size, 2, 2)
            norm_factor = torch.sqrt(torch.sum((xy_iod[:, 0, :] - xy_iod[:, 1, :]) ** 2, dim=-1))  # (batch_size,)
            nme = (errors / norm_factor.unsqueeze(-1)).mean(dim=-1)
            nme = nme.numpy()
            nme_vals.append(nme[nme < 0.2])
    vals = np.concatenate(nme_vals, axis=0)
    checkpoint_nmes[global_step] = [np.mean(vals), np.std(vals)]
    checkpoint_nmfs[global_step] = config.nmf
       

In [ ]:
checkpoint_steps = np.array(list(checkpoint_nmes.keys()))
checkpoint_sorted = np.argsort(checkpoint_steps)
checkpoint_steps = checkpoint_steps[checkpoint_sorted]
checkpoint_vals = np.array(list(checkpoint_nmes.values()))[checkpoint_sorted]
checkpoint_nmfs_arr = np.array(list(checkpoint_nmfs.values()))[checkpoint_sorted]

best_idx = list(checkpoint_steps).index(162800)
# best_idx = np.argmin(checkpoint_vals[:, 0])
print(f"Best checkpoint: Step {checkpoint_steps[best_idx]} with NME {checkpoint_vals[best_idx, 0]:.6f} ± {checkpoint_vals[best_idx, 1]:.6f}")
print(f"NMF for best checkpoint: {checkpoint_nmfs_arr[best_idx]:.6f}")

print(checkpoint_steps[np.argsort(checkpoint_vals[:, 0])[:10]])

plt.plot(checkpoint_steps, checkpoint_vals[:, 0])
plt.plot([checkpoint_steps[best_idx], checkpoint_steps[best_idx]], [np.min(checkpoint_vals[:, 0]), np.max(checkpoint_vals[:, 0])], 'r--')
# plt.fill_between(checkpoint_steps, checkpoint_vals[:, 0] - checkpoint_vals[:, 1], checkpoint_vals[:, 0] + checkpoint_vals[:, 1], alpha=0.5)

plt.figure()
plt.plot(checkpoint_steps, checkpoint_nmfs_arr)
plt.plot([checkpoint_steps[best_idx], checkpoint_steps[best_idx]], [np.min(checkpoint_nmfs_arr), np.max(checkpoint_nmfs_arr)], 'r--')
best_nmf_idx = np.argmin(checkpoint_nmfs_arr)
print(f"Best checkpoint: Step {checkpoint_steps[best_nmf_idx]} with NMF {checkpoint_nmfs_arr[best_nmf_idx]:.6f}")
print(checkpoint_steps[np.argsort(checkpoint_nmfs_arr)[:10]])

## Sample Evaluations

In [ ]:
batch = datasets.wflw_test[370]
cv2.imwrite("wflw_test_sample_370.png", cv2.cvtColor((batch["image"].numpy().transpose(1, 2, 0) * 255).clip(0, 255), cv2.COLOR_RGB2BGR))

In [ ]:
# Load 10 samples from the dataset
sel_data = datasets.wflw_test  # or wflw_train, wflw_test, wflw_v_frames

sample_indices = np.random.choice(len(sel_data), 10, replace=False)
# sample_indices = [370]
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
scatter_plots = []
print(sample_indices)

model.eval()

with torch.no_grad():
    predictions = []
    for i in sample_indices:
        sample = sel_data[i]
        sample = sel_data.wrap(sample, copy=True, device=device)
        print(sample.get_scales())

        p: LandmarkPrediction
        p = model(sample.images.unsqueeze(0), model_queries.get(DatasetName.WFLW).unsqueeze(0).to(device), iterations=3, store_similarity_maps=True)
        predictions.append(
            (p[0].to_mean_max_variance().cpu().numpy(), sample.images, sample.labels)
        )

vmin = min(pred[0][:, 2].min() for pred in predictions)
vmax = max(pred[0][:, 2].max() for pred in predictions)

for i, (ax, (preds, image, labels)) in enumerate(zip(axes.flatten(), predictions)):
    img_np = image.permute(1, 2, 0).cpu().numpy()
    labels = labels.cpu().numpy()
    
    ax.imshow(img_np.clip(0, 1))
    ax.scatter(
        labels[:, 0], labels[:, 1], c='blue', s=10, label='Ground Truth'
    )
    sc = ax.scatter(
        preds[:, 0], preds[:, 1], c=preds[:, 2], s=10,
        vmin=vmin, vmax=vmax, cmap='RdYlGn_r', label='Predicted'
    )
    scatter_plots.append(sc)
    ax.set_title(f"Sample {i}")
    ax.axis('off')

fig.tight_layout(rect=[0, 0, 0.95, 1]) # type: ignore
cbar = fig.colorbar(scatter_plots[0], ax=axes,
                    orientation='vertical', fraction=0.02, pad=0.04)
cbar.set_label('Confidence')
plt.show()

## Test Iterations

In [ ]:
import torchvision.transforms.v2.functional as TF

video = "/home/dogsch/dev/datasets/20251005_165723.mp4"
frames_iter, info = utils.decode_video_frames(video, nframes=1, start=100)
im = list(frames_iter)[0]

im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
im = torch.from_numpy(im).permute(2, 0, 1).unsqueeze(0).float() / 255.0
im = TF.pad(im, [0, 0, 0, 100])
im = TF.center_crop(im, 1000) # type: ignore
im_np_video_iter = cv2.cvtColor((im.squeeze(0).permute(1,2,0) * 255.0).to(torch.uint8).numpy(), cv2.COLOR_RGB2BGR)
im_det_video_iter = TF.resize(im, [224, 224]).to(device)

In [ ]:
fig, ax = plt.subplots(5, 5, figsize=(20, 20))
model.eval()

starting_iterations = 1
with torch.no_grad():
    for i, ax in enumerate(ax.flatten()):
        p: LandmarkPrediction
        p = model(im_det_video_iter, queries_opt.get(DatasetName.WFLW).unsqueeze(0).to(device), iterations=i + starting_iterations)
        p_np = p[0].to_mean_max_variance().cpu().numpy() # type: ignore

        ax.scatter(
            p_np[:, 0] / 224.0 * im_np_video_iter.shape[1], p_np[:, 1] / 224 * im_np_video_iter.shape[0],
            c=p_np[:, 2], s=10,
            cmap='RdYlGn_r', label='Predicted'
        )
        ax.set_title(str(i + starting_iterations))

        ax.imshow(cv2.cvtColor(im_np_video_iter, cv2.COLOR_BGR2RGB))
        ax.axis('off')

## Test on Video

In [ ]:
import torchvision.transforms.v2.functional as TF


# global_step, config = load(config.ckpts_dir() / "10900.pth", model, optimizer)

def create_video(name: str, m=model, nframes = None, encode=True):
    landmarks = []
    video = "/home/dogsch/dev/datasets/20251005_165723.mp4"
    frames_iter, info = utils.decode_video_frames(video, nframes=nframes)
    im_size = 224
    crop_size = 1000
    m.eval()
    
    data = {
        "last_hidden_state": None,
        "last_preds": None
    }

    def get_iter(frames_iter, m):
        frames_iter = tqdm(frames_iter, total=info.nframes)
        frames_iter.refresh()
        for frame in frames_iter:
            # Convert BGR frame (H, W, 3) to RGB tensor (1, 3, H, W)
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0).float() / 255.0
            image = TF.pad(image, [0, 0, 0, 100])
            image = TF.center_crop(image, crop_size) # type: ignore
            image_det = TF.resize(image, [im_size, im_size]).to(device)
            
            # queries = dense_points.to(device).unsqueeze(0)
            queries = model_queries.get(DatasetName.WFLW).clone().unsqueeze(0)
            # nqueries = queries.shape[-2]
            # queries_dup = torch.zeros((*queries.shape[:-2], nqueries*2-1, 3))
            # queries_dup[:, ::2, :] = queries[:, :, :]
            # for i in range(nqueries-1):
            #     queries_dup[:, 1 + i*2, :] = queries[:, i:i+2, :].mean(dim=1)

            # queries = queries_dup # Shape: (1, num_queries, 3)
            with torch.no_grad():
                # Get predictions in pixel coordinates for visualization
                predictions: LandmarkPrediction
                predictions, last_hidden_state = m(image_det, queries.to(device), iterations=1, return_hidden_state=True,

                prefill_starting_landmarks=data["last_preds"], prefill_hidden_state=data["last_hidden_state"])
                if data["last_hidden_state"] is None:
                    data["last_hidden_state"] = last_hidden_state
                else:
                    data["last_hidden_state"][...] = last_hidden_state
                if data["last_preds"] is None:
                    data["last_preds"] = predictions # type: ignore
                else:
                    data["last_preds"].mean[...] = predictions.mean
                    data["last_preds"].cov.params[...] = predictions.cov.params
                    
                res = predictions[0].to_mean_max_variance().cpu().numpy()
            landmarks.append(res)
            
            # frame_np = frame.copy()
            # frame_np = image_det[0].cpu().permute(1, 2, 0).numpy()
            frame_np = cv2.cvtColor((image[0] * 255).to(torch.uint8).permute(1, 2, 0).numpy(), cv2.COLOR_RGB2BGR)
            for j, (x, y, var) in enumerate(res):
                # Map coordinates back to original frame size
                x = int(max(min(round(x * crop_size / im_size), crop_size - 1), 0))
                y = int(max(min(round(y * crop_size / im_size), crop_size - 1), 0))

                # Visualize with confidence-based color
                confidence_norm = min(1.0, var / 20.0)  # Normalize confidence for visualization
                color_intensity = int(255 * confidence_norm)
                color = (0, 255 - color_intensity, color_intensity)  # Green to red based on confidence
                
                cv2.circle(frame_np, (x, y), 3, color, 2)
            yield frame_np
            
    if encode:
        vp, _ = utils.create_video_from_images_cached(get_iter(frames_iter, m), fps=int(info.fps//2), filename=name, force=True)
        return vp, landmarks
    else:
        return list(get_iter(frames_iter, m))

vp, points = create_video("ba19", m=model, encode=True)
# img = create_video("v2", nframes=1, encode=False)[0]
# Image(utils.png_encode(img))